# Task 8 - Word Embedding (BoW / fastText) + SVM

So với TF-IDF (có trọng số theo độ hiếm của từ), notebook này thử 2 cách biểu diễn khác: Bag-of-Words thuần (chỉ đếm từ, không trọng số) và embedding từ đã huấn luyện sẵn (fastText, lấy trung bình vector các từ trong câu). Cả hai vẫn dùng SVM để phân loại, chỉ khác phần biểu diễn đầu vào.

In [ ]:
!pip install -q scikit-learn pandas gensim

In [ ]:
!git clone -q https://github.com/hoanhh4977/SemEval2027.git
%cd SemEval2027/

In [ ]:
import pandas as pd

data = pd.read_csv("data/task8/train/StereoQueerEval_EN_training.tsv", sep="\t")
data.head()

## Chuẩn bị nhãn cho 3 subtask

In [ ]:
# Subtask A - Stereotype Detection (nhị phân)
label2id_A = {"yes": 0, "no": 1}
id2label_A = {v: k for k, v in label2id_A.items()}
data["label_A"] = data["stereotype"].map(label2id_A)

# Subtask B - Hate Speech Classification (3 lớp)
label2id_B = {"no": 0, "yes_implicit": 1, "yes_explicit": 2}
id2label_B = {v: k for k, v in label2id_B.items()}
data["label_B"] = data["hate_speech"].map(label2id_B)

# Subtask C - Target & Identity Identification (multilabel, chỉ trên comment hateful)
def parse_target(x):
    if x == "none":
        return []
    scope, identities = x.split("_", 1)
    return [scope] + identities.split(",")

data["target_labels"] = data["target"].apply(parse_target)
data_c = data[data["hate_speech"] != "no"].reset_index(drop=True)
print(f"Subtask C chỉ dùng {len(data_c)}/{len(data)} comment hateful")

## Chia Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

train_A, test_A = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label_A"])
train_B, test_B = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label_B"])
train_C, test_C = train_test_split(data_c, test_size=0.2, random_state=42)

print(f"A: {len(train_A)}/{len(test_A)}  B: {len(train_B)}/{len(test_B)}  C: {len(train_C)}/{len(test_C)}")

## Cách 1: Bag-of-Words (CountVectorizer) + SVM

Giống TF-IDF nhưng chỉ đếm số lần xuất hiện của từ, không nhân thêm trọng số IDF. Dùng chung một hàm huấn luyện cho cả 3 subtask.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, hamming_loss

def run_svm_subtask(train_df, test_df, label_col, vectorizer, target_names=None, multilabel=False):
    X_train = vectorizer.fit_transform(train_df["yt_comment"])
    X_test = vectorizer.transform(test_df["yt_comment"])

    if multilabel:
        mlb = MultiLabelBinarizer()
        y_train = mlb.fit_transform(train_df[label_col])
        y_test = mlb.transform(test_df[label_col])
        print("Các nhãn:", list(mlb.classes_))

        clf = OneVsRestClassifier(LinearSVC(class_weight="balanced"))
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        print(f"F1-micro: {f1_score(y_test, pred, average='micro'):.3f}  "
              f"F1-macro: {f1_score(y_test, pred, average='macro'):.3f}  "
              f"Hamming loss: {hamming_loss(y_test, pred):.3f}")
        return clf, mlb

    clf = LinearSVC(class_weight="balanced")
    clf.fit(X_train, train_df[label_col])
    pred = clf.predict(X_test)
    print(classification_report(test_df[label_col], pred, target_names=target_names))
    return clf, vectorizer

In [ ]:
print("=== Subtask A ===")
run_svm_subtask(
    train_A, test_A, "label_A",
    CountVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    target_names=[id2label_A[0], id2label_A[1]],
)

print("=== Subtask B ===")
run_svm_subtask(
    train_B, test_B, "label_B",
    CountVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    target_names=[id2label_B[i] for i in sorted(id2label_B)],
)

print("=== Subtask C ===")
run_svm_subtask(
    train_C, test_C, "target_labels",
    CountVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000),
    multilabel=True,
)

## Cách 2: fastText (pretrained word embedding) + SVM

Khác với BoW/TF-IDF (đếm từ xuất hiện trong chính tập dữ liệu), fastText là embedding đã được huấn luyện sẵn trên một kho ngữ liệu khổng lồ — mỗi từ có một vector 300 chiều mang sẵn ý nghĩa ngữ cảnh. Ta lấy trung bình vector của các từ trong một câu để ra vector đại diện cho cả câu. Model tải về khá nặng (~1GB); nếu muốn nhanh hơn có thể đổi tên model sang `"glove-wiki-gigaword-100"` (~128MB, cách dùng y hệt).

In [ ]:
import gensim.downloader as api

ft_model = api.load("fasttext-wiki-news-subwords-300")

In [ ]:
import numpy as np

def embed_mean(texts, model, dim=300):
    vectors = []
    for text in texts:
        tokens = text.lower().split()
        word_vecs = [model[w] for w in tokens if w in model]
        vectors.append(np.mean(word_vecs, axis=0) if word_vecs else np.zeros(dim))
    return np.array(vectors)

def run_svm_subtask_emb(train_df, test_df, label_col, embed_fn, target_names=None, multilabel=False):
    X_train = embed_fn(train_df["yt_comment"])
    X_test = embed_fn(test_df["yt_comment"])

    if multilabel:
        mlb = MultiLabelBinarizer()
        y_train = mlb.fit_transform(train_df[label_col])
        y_test = mlb.transform(test_df[label_col])
        print("Các nhãn:", list(mlb.classes_))

        clf = OneVsRestClassifier(LinearSVC(class_weight="balanced"))
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        print(f"F1-micro: {f1_score(y_test, pred, average='micro'):.3f}  "
              f"F1-macro: {f1_score(y_test, pred, average='macro'):.3f}  "
              f"Hamming loss: {hamming_loss(y_test, pred):.3f}")
        return clf, mlb

    clf = LinearSVC(class_weight="balanced")
    clf.fit(X_train, train_df[label_col])
    pred = clf.predict(X_test)
    print(classification_report(test_df[label_col], pred, target_names=target_names))
    return clf

In [ ]:
embed_fn = lambda texts: embed_mean(texts, ft_model)

print("=== Subtask A ===")
run_svm_subtask_emb(train_A, test_A, "label_A", embed_fn, target_names=[id2label_A[0], id2label_A[1]])

print("=== Subtask B ===")
run_svm_subtask_emb(train_B, test_B, "label_B", embed_fn, target_names=[id2label_B[i] for i in sorted(id2label_B)])

print("=== Subtask C ===")
run_svm_subtask_emb(train_C, test_C, "target_labels", embed_fn, multilabel=True)